\begin{center}
\huge \textbf{Relatório de Avaliação e Benchmark de Modelos de Embedding e Estratégias de Chunking para Dados Normativos (PCDF) }
\end{center}

\vspace{1cm}

**Autores:**

* Lucia Magalhães
* Magno Santana
* Tiago Fialho
* Thiago Lobo

\vspace{0.5cm}

# Introdução

Este relatório apresenta a análise comparativa de diferentes modelos de representação vetorial (*embeddings*) e técnicas de fragmentação (*chunking*) aplicadas à busca semântica em atos normativos e procedimentos operacionais da **Polícia Civil do Distrito Federal (PCDF)**. Os testes utilizaram como base o Procedimento Operacional Padrão (POP) de acolhimento de vítimas de violência doméstica.

O objetivo deste benchmark é identificar a melhor configuração de recuperação (*Retrieval*) para dar suporte a um sistema de Inteligência Artificial do tipo RAG (Geração Aumentada por Recuperação) em infraestrutura local.

# Ambiente de Execução e Hardware
Os testes de desempenho, vetorização e avaliação do benchmark foram executados em uma estação de trabalho **HP Z4 G4 Workstation** com as seguintes especificações físicas relevantes:

* **Processador (CPU):** Intel Xeon W-2175 (18 cores / 28 threads @ 4.00 GHz)
* **Placa de Vídeo (GPU):** NVIDIA Quadro P4000 (8 GB GDDR5 VRAM)
* **Memória RAM:** 64 GB DDR4 (ECC)
* **Sistema Operacional:** Microsoft Windows 10 Pro for Workstations (64-bit)

# Visão Geral do Estudo e Stack Tecnológica
Para a construção e avaliação do pipeline, foram empregadas as seguintes ferramentas e bibliotecas fundamentais:

* **Orquestração e Recuperação (RAG):** `haystack-ai`, `opensearch-haystack` e `ollama-haystack` para a estruturação dos pipelines de busca densa e comunicação com o OpenSearch e modelos locais no Ollama.
* **Busca Semântica Baseada em Grafos:** `lightrag-hku` e `networkx` para a indexação, modelagem relacional e recuperação estruturada em grafos de conhecimento.
* **Modelos e Vetorização (Embeddings):** `sentence-transformers`, `langchain-huggingface`, `langchain-ollama` e `openai` para o gerenciamento e inferência dos modelos de embedding locais e remotos.
* **Extração e Processamento de Documentos:** `docling`, `pymupdf` e `pandas` para parsing de PDFs normativos, OCR e estruturação dos dados brutos em *chunks*.
* **Avaliação de Métricas e Observabilidade:** `ragas` para a mensuração de *Context Precision* e *Context Recall*, e `langfuse` / `langfuse-haystack` para o rastreamento (*tracing*) detalhado das chamadas da LLM e dos nós de busca.
* **Análise Visual e Apresentação de Dados:** `matplotlib` e `seaborn` para a geração dos gráficos de radar, barras e matrizes comparativas de desempenho.

# Metodologia de Avaliação e Métricas

A bancada de testes avaliou o desempenho dos modelos em cenários de corte de documentos (K = 3, K = 5 e K = 10), além de analisar o impacto de estratégias de fragmentação e reordenação (*rerank*), utilizando as seguintes métricas de qualidade e eficiência:

*   **Hit@K (Sensibilidade de Sucesso):** Taxa de vezes em que o documento correto foi recuperado entre as K primeiras posições.
*   **Recall@K / Context Recall:** Proporção de documentos relevantes recuperados em relação ao total de relevantes existentes.
*   **Context Precision:** Mede se os chunks recuperados contêm o conteúdo relevante necessário para responder à consulta de forma precisa.
*   **MRR (Mean Reciprocal Rank):** Avalia a posição exata da resposta correta no ranking (1.0 se estiver em primeiro lugar).
*   **NDCG@K (Ganho Cumulativo Descontado Normalizado):** Mede a qualidade da ordenação no topo, penalizando o modelo se a resposta correta ficar abaixo do ruído.
*   **Latência (segundos):** Tempo físico de resposta para computar os embeddings, execuções e ranqueamento.

# Resultados Intercalados por Parâmetro de Recuperação (K)

## Cenário de Restrição Crítica de Contexto (K = 3)

Neste cenário, avalia-se o comportamento do sistema quando limitamos estritamente o envio de informações ao modelo de linguagem a apenas 3 trechos normativos. É o cenário ideal para economizar contexto do LLM, mas exige precisão cirúrgica do modelo de embedding.

| Distribuição de Métricas de Qualidade e Latência (K = 3) | Área de Cobertura Semântica - Radar (K = 3) |
| :---: | :---: |
| ![Gráfico de Barras K=3](output/0_k3_grafico_barras.png) | ![Gráfico de Radar K=3](output/0_k3_grafico_radar.png) |
| *Figura 1a: Métricas de qualidade versus latência física para K = 3.* | *Figura 1b: Comparação de área de separabilidade e ordenação para K = 3.* |

**Análise dos Resultados (K = 3):**

*   **A Superioridade do `bge-m3`:** Mesmo com pouquíssimas chances de trazer o documento correto, o `bge-m3` se destaca de forma isolada no **NDCG@3** e no **MRR**, como ilustrado pelo bico destacado na direção de MRR no gráfico de radar (Figura 1b).
*   **Comportamento do `granite-embedding:278m`:** Mostra-se altamente competitivo, superando concorrentes de grande porte como `bge-large` e `mxbai-embed-large` em quase todas as métricas de qualidade, mantendo uma das menores latências do teste (~5 segundos).
*   **O Gargalo do `paraphrase-multilingual`:** Embora tenha boa sensibilidade (Hit@3), sua latência ultrapassa 9 segundos, inviabilizando-o para sistemas em tempo real sem hardware dedicado.

## O Ponto de Equilíbrio Operacional (K = 5)

Aumentar o limite para 5 documentos busca o ponto de equilíbrio (*sweet spot*) entre oferecer contexto suficiente para o LLM responder com segurança e não sobrecarregar a janela de texto com informações redundantes.

| Distribuição de Métricas de Qualidade e Latência (K = 5) | Área de Cobertura Semântica - Radar (K = 5) |
| :---: | :---: |
| ![Gráfico de Barras K=5](output/0_k5_grafico_barras.png) | ![Gráfico de Radar K=5](output/0_k5_grafico_radar.png) |
| *Figura 2a: Métricas de qualidade versus latência física para K = 5.* | *Figura 2b: Comparação de área de separabilidade e ordenação para K = 5.* |

**Análise dos Resultados (K = 5):**

*   **Evolução Semântica:** Há um salto nítido no **Hit@5** e **Recall@5** para todos os modelos. O `bge-m3` e o `paraphrase-multilingual` empatam na capacidade bruta de localizar a regra jurídica no acervo, o que se traduz em áreas quase sobrepostas no topo do radar (Figura 2b).
*   **Consolidação do Granite:** O `granite-embedding:278m` consolida sua posição como a melhor alternativa de otimização: entrega métricas de qualidade extremamente próximas ao líder absoluto, com uma latência significativamente menor (~4.5 segundos).
*   **Estabilidade de Latência:** Note que a latência de todos os modelos permanece praticamente a mesma em comparação com K=3. Isso ocorre porque a maior parte do esforço computacional local do Ollama está na vetorização inicial dos textos, e não na ordenação matemática dos vetores na memória.

## Cenário de Contexto Expandido (K = 10)

Neste cenário, permitimos que até 10 trechos normativos sejam recuperados para alimentar a resposta. Avalia-se se o ganho marginal em sensibilidade compensa o risco de injetar ruído na geração final.

| Distribuição de Métricas de Qualidade e Latência (K = 10) | Área de Cobertura Semântica - Radar (K = 10) |
| :---: | :---: |
| ![Gráfico de Barras K=10](output/0_k10_grafico_barras.png) | ![Gráfico de Radar K=10](output/0_k10_grafico_radar.png) |
| *Figura 3a: Métricas de qualidade versus latência física para K = 10.* | *Figura 3b: Comparação de área de separabilidade e ordenação para K = 10.* |

**Análise dos Resultados (K = 10):**

*   **Retornos Decrescentes de Sensibilidade:** Embora o **Hit@10** chegue a patamares elevados (próximos a 0.8 para Granite e BGE), a diferença em relação a K=5 é marginal. Isso indica que expandir o contexto para 10 documentos traz pouca informação nova e relevante, servindo apenas para inflar o prompt do LLM.
*   **Liderança em Ordenação Mantida:** O `bge-m3` continua soberano em manter as respostas corretas no topo. No gráfico de radar (Figura 3b), a linha laranja engloba quase todas as outras áreas nas direções de **MRR** e **NDCG@10**, provando sua robustez matemática.

# Avaliação Geral dos Modelos de Embedding Testados

1.  **`bge-m3` (Líder Absoluto de Qualidade):** A escolha ideal para precisão jurídica. Sua capacidade de ordenação (**MRR** e **NDCG**) é imbatível. Lida de forma nativa e precisa com siglas ("IML", "DEAM", "PCDF") e terminologias formais de atos normativos em português brasileiro.
2.  **`granite-embedding:278m` (A Surpresa de Custo-Benefício):** Desenvolvido pela IBM, é um modelo extremamente leve (278 milhões de parâmetros) que entrega **90% da precisão** do `bge-m3`, com uma latência consistentemente menor e menor consumo de recursos físicos (memória e CPU).
3.  **`paraphrase-multilingual` (Inviável Localmente):** Embora possua excelente capacidade de capturar o sentido de termos formais de leis e regras, sua execução no Ollama local gerou um gargalo crítico de latência (~10 segundos), tornando-o inadequado para ambientes de produção.

# Análise de Estratégias de Chunking e Impacto do Rerank (Fase 1)

Nesta etapa, avaliamos como a fragmentação do POP e a aplicação de um modelo de reordenação (*rerank* com `BAAI/bge-reranker-base`) impactam a recuperação em termos de **Context Precision** e **Context Recall**.

| Comparativo Consolidado de Precisão e Recall com e sem Rerank |
| :---: |
| ![Gráfico Consolidado de Precision e Recall](output/1_comparativo_geral_precisao_recall.png) |
| *Figura 4: Comparativo consolidado de Context Precision e Context Recall por estratégia de chunking (com e sem rerank).* |

## Análise dos Resultados e Comportamento do Rerank

*   **A Superioridade do Chunking Fixo:** A estratégia de chunking **fixo sem reranking** atingiu o melhor desempenho geral do benchmark, destacando-se especialmente no indicador de **Context Recall** (superando um escore médio de 0.64), conforme evidenciado na Figura 4.
*   **O Impacto do Rerank:** Contrariando a premissa de ganho absoluto de ordenação, a inclusão do componente de *rerank* (`BAAI/bge-reranker-base`) reduziu marginalmente os escores de precisão e revocação na maioria das variações de chunking testadas. Isso indica que o retriever denso inicial já recupera os trechos de forma otimizada para este corpus normativo específico, e o cross-encoder acabou penalizando recortes que necessitavam de amplitude contextual.
*   **Comportamento das Demais Estratégias:** As abordagens hierárquica, recursiva, semântica e de janela de sentença apresentaram desempenhos intermediários e inferiores ao chunking fixo básico, demonstrando que fragmentações complexas ou excessivamente picotadas diluem o contexto jurídico necessário.

# Avaliação Comparativa de Técnicas de Recuperação (OpenSearch vs. LightRAG)

Nesta etapa final, avaliamos o desempenho prático de diferentes estratégias de recuperação de texto sobre o corpus normativo da PCDF, utilizando o conjunto de perguntas de validação e o algoritmo campeão de *chunking* fixo. Foram testadas abordagens baseadas em expansão de consultas no OpenSearch (*Baseline*, *Multi-Query*, *RAG Fusion* e *Step-Back*) e uma abordagem de recuperação estruturada baseada em grafos de conhecimento (*LightRAG* no modo híbrido).

| Comparativo de Desempenho RAGAS por Técnica de Recuperação |
| :---: |
| ![Gráfico Comparativo de Técnicas RAGAS](output/3_comparativo_tecnicas_ragas.png) |
| *Figura 5: Comparativo consolidado de Context Precision e Context Recall entre as técnicas no OpenSearch e a abordagem baseada em Grafo (LightRAG).* |

## Análise Comparativa e Qualitativa dos Resultados

Para compreender as discrepâncias observadas no gráfico entre as pontuações do Ragas e a real utilidade prática das respostas, a tabela a seguir apresenta a comparação direta de três casos de teste representativos extraídos do conjunto de validação:

| ID / Pergunta | Gabarito Institucional | Resposta OpenSearch Baseline | Resposta LightRAG (Grafo) |
| :--- | :--- | :--- | :--- |
| **Q1:** Qual é o objetivo principal do protocolo de acolhimento de mulheres vítimas de violência doméstica e familiar nas Delegacias de Polícia? | O objetivo principal é padronizar o procedimento de acolhimento sob a perspectiva de gênero para coletar e formar as provas necessárias à investigação policial e à apreciação de medidas protetivas de urgência, de modo a impedir a revitimização da vítima. | O objetivo principal do protocolo de acolhimento de mulheres vítimas de violência doméstica e familiar nas Delegacias de Polícia é padronizar o procedimento de acolhimento dessas mulheres, coletar e formar as provas necessárias à apreciação das medidas protetivas de urgência e à investigação policial, com o fim de impedir a revitimização. | O objetivo principal do protocolo [...] é garantir a assistência adequada e o atendimento humanizado às vítimas, priorizando sua segurança, bem-estar e direitos. O protocolo visa fornecer uma resposta eficaz e respeitosa [...]. O protocolo inclui procedimentos específicos [...]: 1. Acolhimento imediato e respeitoso; 2. Identificação das necessidades [...]; 3. Realização de exames médicos e psicológicos [...]; 4. Coleta de provas [...]; 5. Encaminhamento [...]; 6. Fornecimento de orientações [...]. |
| **Q11:** Quais sistemas de consulta devem ser obrigatoriamente pesquisados para todos os envolvidos no registro? | Devem ser consultados os sistemas MILLENIUM, PROCED e SIIC de todos os envolvidos, incluindo as testemunhas. Os resultados devem ser impressos e anexados à ocorrência e ao requerimento de medidas protetivas. | Os sistemas MILLENIUM, PROCED e SIIC devem ser obrigatoriamente pesquisados para todos os envolvidos na ocorrência. (Trecho 4.II) | De acordo com o documento fornecido, os sistemas de consulta que devem ser obrigatoriamente pesquisados para todos os envolvidos no registro incluem: 1. SISBIN (Sistema de Informações sobre Identificação Civil) 2. SPC (Sistema de Proteção ao Crédito) 3. CPF (Cadastro de Pessoas Físicas) 4. RG (Registro Geral) [...] |
| **Q27:** Para onde a vítima de violência sexual deve ser encaminhada para prevenção do HIV e qual o prazo e duração recomendados para esse tratamento? | A vítima deve ser encaminhada ao PAV (Programa de Pesquisa, Assistência e Vigilância em Violência) da Secretaria de Saúde para a Profilaxia Pós-Exposição (PEP). O tratamento dura 28 dias e deve ser iniciado preferencialmente nas primeiras 2 horas e no limite máximo de até 72 horas após a exposição. | A vítima de violência sexual deve ser encaminhada ao Programa de Pesquisa, Assistência e Vigilância em Violência - PAV, da Secretaria de Estado de Saúde, para submissão à Profilaxia Pós-Exposição - PEP ao HIV. O tratamento tem duração de 28 dias e deve ser iniciado o mais rápido possível, preferencialmente nas primeiras duas horas após a exposição e no máximo em até 72 horas. | Lamento, mas não posso fornecer informações sobre procedimentos médicos específicos. Se você ou alguém que conhece está procurando ajuda após uma experiência de violência sexual, é importante buscar orientação profissional imediatamente [...]. |

## Análise Crítica e Viés de Avaliação

*   **Desempenho das Abordagens Lexicais/Densas (OpenSearch):** As técnicas executadas sobre o OpenSearch com *chunking* fixo obtiveram os maiores escores de **Context Precision** (destaque para o *Baseline* com $0.933$) e **Context Recall** ($0.672$). O *Baseline* se destacou por recuperar e reproduzir com precisão quase literal os trechos normativos (como demonstrado nos exemplos Q1 e Q11), atendendo perfeitamente ao rigor do gabarito sintético.
*   **O Comportamento do LightRAG (Grafo) e o Viés do Ragas:** A abordagem `LightRAG` registrou pontuações significativamente menores no Ragas (`Precision`: $0.433$, `Recall`: $0.183$). Contudo, a análise qualitativa das respostas revela três fatores determinantes para essa penalização:
    1.  **Prolixidade e Síntese Conceitual:** Em perguntas gerais (como na Q1), o LightRAG sintetiza o contexto do grafo em listas estruturadas e explicativas. Embora essa abordagem seja operacionalmente rica e didática para um policial na ponta, a métrica do Ragas penaliza a resposta por afastar-se da literalidade enxuta do gabarito.
    2.  **Alucinação por Generalização em Termos Específicos:** Em consultas sobre siglas ou sistemas normativos locais rígidos (como na Q11), o LightRAG tendeu a inferir ou mapear entidades genéricas do seu conhecimento base (mencionando erroneamente sistemas como *SISBIN* ou *SPC* em vez de *MILLENIUM/PROCED/SIIC*).
    3.  **Falsos Positivos de Moderadores de Segurança (Safety Guardrails):** Em perguntas que continham termos sensíveis relacionados a crimes sexuais e procedimentos médicos de urgência (como a Q27, além das perguntas Q13, Q21, Q22 e Q26), a LLM acoplada ao LightRAG ativou indevidamente recusas de segurança (*"Lamento, mas não posso fornecer informações..."*), zera a revocação nesses itens e derrubando a média geral da técnica.

# Conclusão e Diretrizes Finais de Arquitetura para a PCDF

O desenvolvimento e a validação empírica deste pipeline de Recuperação Aumentada por Geração (RAG) para os atos normativos e procedimentos operacionais da Polícia Civil do Distrito Federal (PCDF) permitiram estabelecer um panorama técnico claro sobre a infraestrutura de inteligência artificial local. A partir da mensuração sistemática via *Ragas* e da auditoria qualitativa dos logs e respostas, consolidamos as seguintes diretrizes de arquitetura:

*   **Fundação de Vetorização (Embeddings):** O modelo `bge-m3` provou ser o líder absoluto de qualidade e ordenação, lidando com precisão nativa com siglas e terminologias formais do corpus normativo. Como alternativa balanceada para servidores locais com restrição de hardware, o `granite-embedding:278m` destacou-se com excelente custo-benefício.
*   **Estratégia de Fragmentação (Chunking):** A estratégia de *chunking* **fixo puro** superou abordagens semânticas, hierárquicas e de janela de sentença, preservando melhor as unidades lógicas e a estrutura rígida dos formulários institucionais sem o ônus de latências elevadas.
*   **Abstenção de Rerank Desnecessário:** A aplicação de modelos de reordenação (*rerank* com `BAAI/bge-reranker-base`) demonstrou retornos marginais ou negativos, evidenciando que o recuperador denso inicial já entrega um agrupamento otimizado.
*   **Ajuste Fino de Observabilidade e Guardrails:** Identificou-se que arquiteturas de monitoramento (como *Langfuse*) devem ter a captura de argumentos de baixa camada (ex: vetores e nós de grafos) filtrada para evitar gargalos de I/O em payloads de larga escala. Da mesma forma, os *prompts* de sistema do gerador devem desativar alinhamentos de recusa genéricos que bloqueiem termos policiais ou periciais legítimos presentes no corpus.
*   **Estratégia de Recuperação e Consulta (OpenSearch vs. Grafos):** A busca densa otimizada no OpenSearch (*Baseline*) garantiu maior precisão factual e conformidade estrita aos procedimentos operacionais e siglas da PCDF. O *LightRAG*, embora promissor para análises relacionais e resumos conceituais amplos, exige maior controle de prompt e alinhamento para evitar alucinações em normas específicas e recusas indevidas em temas sensíveis.

**Configuração Recomendada (Padrão Ouro Institucional):**

*   **Modelo de Embedding:** `bge-m3`
*   **Estratégia de Chunking:** **Fixo** (sem *rerank* adicional)
*   **Parâmetro de Corte ($K$):** $5$
*   **Mecanismo de Recuperação:** Busca densa otimizada (*Baseline* no OpenSearch) para consultas procedimentais diretas, com suporte complementar a arquiteturas em grafo para geração de relatórios investigativos integrados.